# Marine heatwaves and phytoplankton — live demo

**Ocean Hackathon.** Runs end to end in about two minutes with nothing installed locally.

Open in Colab, then `Runtime -> Run all`.

Before first use, replace `YOUR-USER/YOUR-REPO` in the next cell with your repository.

In [ ]:
REPO = "YOUR-USER/YOUR-REPO"  # <-- edit this

import os, sys, subprocess
if not os.path.exists("mhw-phyto"):
    subprocess.run(["git", "clone", "--depth", "1",
                    f"https://github.com/{REPO}.git", "mhw-phyto"], check=True)
os.chdir("mhw-phyto")
sys.path.insert(0, "src")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "xarray", "netCDF4"], check=True)
print("ready")

## 1. The claim we are testing

Marine heatwaves do not have one effect on phytoplankton. In stratified water, heat
cuts off nutrient supply from below and chlorophyll falls. In light- or mixing-limited
water, the same heat can advance and boost a bloom. So the question is not *whether*
heatwaves matter but **where, with what lag, and with what sign**.

## 2. Validation first

Before showing any result, two checks. Our detector must agree with the canonical
implementation of the Hobday et al. (2016) definition, and our response analysis must
recover a response we planted ourselves — including finding *nothing* where we planted
nothing.

In [ ]:
!python -m pytest tests -q
!python scripts/validate_against_reference.py --ref-path vendor | tail -12

## 3. Detection on a synthetic ocean with known ground truth

In [ ]:
import pandas as pd, numpy as np
from mhwphyto import synth, mhw, composite

BASELINE = ("1998-01-01", "2017-12-31")
ds = synth.make_dataset()
time = pd.DatetimeIndex(ds["time"].values)
sst, chl = ds["sst"].values, ds["chl"].values

events, clim, thresh = mhw.detect_grid(sst, time, baseline=BASELINE)
print(f"{len(events)} events over {sst.shape[1]*sst.shape[2]} cells")
print(events.groupby('category_name')[['duration','intensity_max']].agg(['count','mean']))

## 4. The result: chlorophyll response splits by regime

Composite the chlorophyll anomaly on event onset, per cell. Confidence intervals are
bootstrapped over **events**, not days — days inside a heatwave are strongly
autocorrelated and resampling them would manufacture significance.

In [ ]:
import matplotlib.pyplot as plt

chl_anom = composite.log_anomaly(chl, time, baseline=BASELINE)
truth = ds["regime"].values

fig, ax = plt.subplots(figsize=(9, 5))
colors = {"stratified": "tab:red", "light_limited": "tab:blue",
          "insensitive": "0.5"}
seen = set()
for name, (j, i) in [("stratified", (0, 0)), ("light_limited", (4, 0)),
                     ("insensitive", (0, 5))]:
    onsets = events[(events.lat_index == j) & (events.lon_index == i)].start_index
    comp = composite.lag_composite(chl_anom[:, j, i], onsets)
    ax.plot(comp.index, comp["mean"], color=colors[name], lw=2,
            label=f"{name} (planted: {synth.REGIME_TRUTH[name]})")
    ax.fill_between(comp.index, comp["lo"], comp["hi"],
                    color=colors[name], alpha=0.18)
    s = composite.summarise_composite(comp)
    print(f"{name:14s} recovered direction={s['direction']:5s} "
          f"peak={s['peak_response']:+.3f} at lag {s['peak_lag']:3d} d")

ax.axhline(0, color="k", lw=0.8)
ax.axvline(0, color="k", lw=0.8, ls=":")
ax.set_xlabel("days relative to marine heatwave onset")
ax.set_ylabel("log10 chlorophyll anomaly")
ax.set_title("Regime-dependent chlorophyll response (shading = 95% CI over events)")
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

## 5. Full pipeline, including the forecast and its baselines

A model that loses to persistence is a real finding, so the baselines are computed
first and skill is reported against the better of them.

In [ ]:
!python scripts/run_walking_skeleton.py

In [ ]:
from IPython.display import Image, display
display(Image("outputs/walking_skeleton.png"))

## 6. Swap in real data

Everything above runs on synthetic data with a planted, therefore learnable, response.
Real skill will be lower. To run on observations, upload a CMEMS or NOAA subset and
point the loader at it — no analysis code changes:

```python
from mhwphyto import data
ds = data.load(source="local", path="data/subset.nc")
```

or from the command line:

```bash
python scripts/run_walking_skeleton.py --source local --path data/subset.nc
```

### Caveats to state before a judge asks

- Satellite chlorophyll is a **surface** measurement and is unreliable in turbid coastal water.
- Cloud gaps bias event windows in ocean-colour records.
- Photoacclimation changes the chlorophyll-to-carbon ratio under warming, so a chlorophyll drop is **not** necessarily a biomass drop.